# 05 PSO Hyperparameter Tuning (Improved)

This notebook tunes CNN hyperparameters with PySwarms using an **expanded 6D search space** (filters, kernel_size, dense_units, learning_rate, dropout_rate, num_conv_layers).

**Improvements over previous version:**
- Enhanced CNN with BatchNormalization, Dropout, multi-layer support
- EarlyStopping + ReduceLROnPlateau callbacks
- Log-transform on effort target to handle skew
- 6D PSO search (was 4D) with 15 particles × 25 iterations
- Models saved to `models/` for web app

In [1]:
import json
from pathlib import Path
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

root_dir = Path.cwd()
if not (root_dir / "src").exists() and (root_dir.parent / "src").exists():
    root_dir = root_dir.parent

if not (root_dir / "src").exists():
    raise FileNotFoundError("Could not find project root containing 'src' directory")

sys.path.insert(0, str(root_dir))

from src.cnn_model import build_cnn_regressor, reshape_for_cnn, train_cnn_model
from src.evaluate import evaluate_predictions, save_metrics
from src.feature_engineering import log_transform_target, inverse_log_transform
from src.pso_optimizer import decode_cnn_hyperparameters, tune_cnn_with_pso

processed_dir = root_dir / "data" / "processed"
metrics_dir = root_dir / "results" / "metrics"
models_dir = root_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Load all 3 datasets for evaluation
datasets = {}
for name, fname in [("desharnais", "desharnais_processed.csv"),
                     ("cocomo81", "cocomo81_processed.csv"),
                     ("china", "china_processed.csv")]:
    path = processed_dir / fname
    if path.exists():
        datasets[name] = pd.read_csv(path)

# Primary dataset for PSO tuning
primary_name = "desharnais" if "desharnais" in datasets else list(datasets.keys())[0]
df = datasets[primary_name]
target_col = df.columns[-1]
X = df.drop(columns=[target_col]).values.astype(np.float32)
y = df[target_col].values.astype(np.float32)

# Log-transform effort target
y_log = log_transform_target(y).astype(np.float32)

X_train, X_test, y_train, y_test, y_train_log, y_test_log = train_test_split(
    X, y, y_log, test_size=0.2, random_state=42
)

X_train_cnn = reshape_for_cnn(X_train)
X_test_cnn = reshape_for_cnn(X_test)

print(f"Dataset: {primary_name} | Features: {X.shape[1]} | Samples: {len(X)}")
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Target column: {target_col}")

Dataset: desharnais | Features: 12 | Samples: 81
Train: 64 | Test: 17
Target column: Effort


In [2]:
# --- Baseline CNN (fixed hyperparams, log-transformed target) ---

baseline_cnn = build_cnn_regressor(
    input_length=X_train_cnn.shape[1],
    filters=32,
    kernel_size=3,
    dense_units=64,
    learning_rate=1e-3,
    dropout_rate=0.2,
    num_conv_layers=1,
)

baseline_cnn, baseline_history = train_cnn_model(
    baseline_cnn,
    X_train_cnn, y_train_log,
    X_test_cnn, y_test_log,
    epochs=100,
    batch_size=32,
    verbose=0,
    use_callbacks=True,
)

# Predict in log space, then inverse transform
baseline_preds_log = baseline_cnn.predict(X_test_cnn, verbose=0).ravel()
baseline_preds = inverse_log_transform(baseline_preds_log)

baseline_metrics = evaluate_predictions("cnn_baseline", y_test, baseline_preds)
print(f"Baseline CNN stopped at epoch {len(baseline_history['loss'])}")
baseline_metrics

Baseline CNN stopped at epoch 25


,model,mae,rmse,r2,mape,mdmre,pred25
0,cnn_baseline,4534.115241,5772.218356,-1.611423,99.902987,0.999637,0.0


In [3]:
# --- PSO Optimization (6D search with log-transformed target) ---

def objective_fn(particles: np.ndarray) -> np.ndarray:
    """Evaluate a swarm of particles — returns validation MAE for each."""
    scores = []
    for particle in particles:
        hp = decode_cnn_hyperparameters(particle)
        try:
            model = build_cnn_regressor(
                input_length=X_train_cnn.shape[1],
                filters=hp["filters"],
                kernel_size=hp["kernel_size"],
                dense_units=hp["dense_units"],
                learning_rate=hp["learning_rate"],
                dropout_rate=hp.get("dropout_rate", 0.2),
                num_conv_layers=hp.get("num_conv_layers", 1),
            )
            model, history = train_cnn_model(
                model,
                X_train_cnn, y_train_log,
                X_test_cnn, y_test_log,
                epochs=30,
                batch_size=32,
                verbose=0,
                use_callbacks=True,
            )
            val_mae = min(history["val_mean_absolute_error"])
            scores.append(val_mae)
        except Exception:
            scores.append(1e6)  # penalize failed configs
    return np.array(scores)

# 6D bounds: [filters, kernel_size, dense_units, learning_rate, dropout_rate, num_conv_layers]
lower_bounds = np.array([8,   2,  8,  1e-4, 0.1, 1], dtype=float)
upper_bounds = np.array([128, 7, 256, 1e-2, 0.5, 3], dtype=float)

print("Starting PSO optimization (15 particles × 25 iterations)...")
print("This may take 10-20 minutes on CPU.\n")

best_cost, best_position = tune_cnn_with_pso(
    objective_fn=objective_fn,
    dimensions=6,
    lower_bounds=lower_bounds,
    upper_bounds=upper_bounds,
    n_particles=15,
    iters=25,
)

best_hyperparams = decode_cnn_hyperparameters(best_position)
print("\nBest PSO validation MAE (log space):", round(best_cost, 4))
print("Best hyperparameters:", best_hyperparams)

# Save best hyperparams
hp_path = models_dir / "best_hyperparams.json"
with open(hp_path, "w") as f:
    json.dump(best_hyperparams, f, indent=2)
print(f"Saved hyperparams to {hp_path}")

Starting PSO optimization (15 particles × 25 iterations)...
This may take 10-20 minutes on CPU.



2026-04-17 18:43:59,349 - pyswarms.single.global_best - INFO - Optimize for 25 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|25/25, best_cost=1.14
2026-04-17 19:47:48,537 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 1.140892744064331, best pos: [1.27053893e+02 4.75903443e+00 1.49184861e+02 5.06823772e-03
 3.85492244e-01 2.47703769e+00]



Best PSO validation MAE (log space): 1.1409
Best hyperparameters: {'filters': 127, 'kernel_size': 5, 'dense_units': 149, 'learning_rate': 0.005068237718309163, 'dropout_rate': 0.3854922437060443, 'num_conv_layers': 2}
Saved hyperparams to c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models\best_hyperparams.json


In [4]:
# --- Train final CNN+PSO model with best hyperparams ---

pso_cnn = build_cnn_regressor(
    input_length=X_train_cnn.shape[1],
    filters=best_hyperparams["filters"],
    kernel_size=best_hyperparams["kernel_size"],
    dense_units=best_hyperparams["dense_units"],
    learning_rate=best_hyperparams["learning_rate"],
    dropout_rate=best_hyperparams.get("dropout_rate", 0.2),
    num_conv_layers=best_hyperparams.get("num_conv_layers", 1),
)

pso_cnn, pso_history = train_cnn_model(
    pso_cnn,
    X_train_cnn, y_train_log,
    X_test_cnn, y_test_log,
    epochs=100,
    batch_size=32,
    verbose=0,
    use_callbacks=True,
)

# Predict and inverse transform
pso_preds_log = pso_cnn.predict(X_test_cnn, verbose=0).ravel()
pso_preds = inverse_log_transform(pso_preds_log)

pso_metrics = evaluate_predictions("cnn_pso", y_test, pso_preds)
print(f"CNN+PSO stopped at epoch {len(pso_history['loss'])}")

# Compare
comparison = pd.concat([baseline_metrics, pso_metrics], ignore_index=True).sort_values("rmse")
print("\n--- CNN Baseline vs CNN+PSO ---")
comparison

CNN+PSO stopped at epoch 50

--- CNN Baseline vs CNN+PSO ---


,model,mae,rmse,r2,mape,mdmre,pred25
1,cnn_pso,2574.512419,4139.047751,-0.342742,77.612369,0.532316,29.411765
0,cnn_baseline,4534.115241,5772.218356,-1.611423,99.902987,0.999637,0.000000


In [5]:
# --- Save metrics and models ---

# Save comparison metrics
comparison_path = save_metrics(comparison, file_name="cnn_vs_pso_metrics.csv", output_dir=metrics_dir)
print("Saved comparison metrics to:", comparison_path)

# Save the PSO-tuned CNN model
model_path = models_dir / "cnn_pso_model.h5"
pso_cnn.save(model_path)
print(f"Saved CNN+PSO model to: {model_path}")

# Save baseline CNN model
baseline_path = models_dir / "cnn_baseline_model.h5"
baseline_cnn.save(baseline_path)
print(f"Saved baseline CNN to: {baseline_path}")

# Save training histories for visualization
import json as _json
histories = {
    "baseline": {k: [float(v) for v in vals] for k, vals in baseline_history.items()},
    "pso": {k: [float(v) for v in vals] for k, vals in pso_history.items()},
}
hist_path = models_dir / "training_histories.json"
with open(hist_path, "w") as f:
    _json.dump(histories, f)
print(f"Saved training histories to: {hist_path}")

# Compare against baselines from notebook 03
baseline_file = metrics_dir / "baseline_metrics.csv"
if baseline_file.exists():
    baselines = pd.read_csv(baseline_file)
    full_comparison = pd.concat([baselines, comparison], ignore_index=True).sort_values("rmse")
    print("\n--- Full Model Comparison ---")
    display(full_comparison)
    save_metrics(full_comparison, file_name="full_comparison.csv", output_dir=metrics_dir)
else:
    print("Run 03_baselines.ipynb first for full comparison")

2026-04-17 19:48:04,147 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 


Saved comparison metrics to: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\results\metrics\cnn_vs_pso_metrics.csv
Saved CNN+PSO model to: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models\cnn_pso_model.h5


2026-04-17 19:48:04,342 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 


Saved baseline CNN to: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models\cnn_baseline_model.h5
Saved training histories to: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models\training_histories.json

--- Full Model Comparison ---


,model,mae,rmse,r2,mape,mdmre,pred25
0,linear_regression,1435.054687,1943.914123,NaN,NaN,NaN,NaN
1,xgboost,1693.692017,2245.187743,NaN,NaN,NaN,NaN
2,random_forest,1755.944118,2283.387200,NaN,NaN,NaN,NaN
3,cnn_pso,2574.512419,4139.047751,-0.342742,77.612369,0.532316,29.411765
4,cnn_baseline,4534.115241,5772.218356,-1.611423,99.902987,0.999637,0.000000
